# Precisión Mixta

Refinamiento iterativo de precisión mixta


In [ ]:
"""
PRECISION MIXTA - Refinamiento iterativo para resolver A x = b
================================================================
Idea: DENTRO DE UN MISMO ALGORITMO se combinan varios formatos:

  0) Los datos se GENERAN Y ALMACENAN primero en float16. Simula
     datos que vienen de una fuente muy limitada: un sensor barato,
     un archivo comprimido, hardware de bajo consumo, etc.
  1) Factorizar A -> se hace en la precision mas baja posible que
     soporte la libreria de algebra lineal (rapido y barato)
  2) Calcular el residuo r = b - A x -> se promueve puntualmente a
     float64 (aqui NO se puede ahorrar precision: restar dos numeros
     casi iguales en baja precision destruye la informacion del error)
  3) Corregir la solucion con ese residuo -> precision baja para
     resolver, float64 para acumular

"""

import numpy as np
from scipy.linalg import lu_factor, lu_solve

np.random.seed(0)
n = 400

# ------------------------------------------------------------------
# PASO 0: generar y ALMACENAR el sistema en el formato
# float16. Esta es la representacion "de origen" de los
# datos, tal como si vinieran de una fuente de precision muy limitada.
# ------------------------------------------------------------------
B16 = np.random.randn(n, n).astype(np.float16)

A16 = (B16.astype(np.float32) @ B16.astype(np.float32).T+ n * np.eye(n, dtype=np.float32)).astype(np.float16)

x_true16 = np.random.randn(n).astype(np.float16)

b16 = (A16.astype(np.float32) @ x_true16.astype(np.float32)).astype(np.float16)

print(f"Tamano del sistema: {n} x {n}")
print(f"dtype de A16 (almacenamiento original, formato MAS PEQUENO): {A16.dtype}")
print(f"dtype de b16 (almacenamiento original): {b16.dtype}\n")

# ------------------------------------------------------------------
# Version float64 SOLO como referencia para medir el error real.
# En un caso real esto no existiria; aqui se promueve unicamente para
# tener un "patron de verdad" con el que comparar.
# ------------------------------------------------------------------
A64 = A16.astype(np.float64)
b64 = b16.astype(np.float64)
x_true = x_true16.astype(np.float64)
print("(Solo para comparar resultados, se promueve una copia a float64:")
print(f" A64.dtype = {A64.dtype}, b64.dtype = {b64.dtype})\n")

# ------------------------------------------------------------------
# Version float32: el formato mas pequeno que LAPACK puede factorizar.
# Se promueve desde float16 SOLO para poder llamar a lu_factor/lu_solve.
# ------------------------------------------------------------------
A32 = A16.astype(np.float32)
b32 = b16.astype(np.float32)

print(">> Factorizando A (LAPACK no soporta float16, se promueve el minimo")
print(f"   necesario para poder factorizar: {A32.dtype})")
lu, piv = lu_factor(A32)
print(f"   lu.dtype = {lu.dtype}\n")

# Solucion inicial: se resuelve barato en float32, luego se promueve a float64
# Esta x0 es solo una APROXIMACION del resultado real (x_true), porque
# tanto los datos de origen (float16) como el solve (float32) son de
# precision baja.
x0_32 = lu_solve((lu, piv), b32)
x = x0_32.astype(np.float64)
print(f"Solucion inicial x0 calculada en {A32.dtype}, promovida a {x.dtype}")
print(f"Error de x0 respecto al resultado REAL (x_true): "
      f"{np.linalg.norm(x - x_true):.3e}\n")

print(f"{'iter':>4} | {'dtype(r)':>9} | {'||r||=||b-Ax||':>15} | {'||x - x_true||':>15}")
print("-" * 55)

tol = 1e-10
for it in range(1, 9):
    # ------------------------------------------------------------------
    # PASO 1: calcular el residuo, promoviendo A y b (guardados en
    # float16) a float64 SOLO para esta resta.
    #   r = b64 - A64 @ x
    # r mide "cuanto le falta" a la x actual para satisfacer A x = b
    # con la precision mas alta posible. Si se hiciera esta resta en
    # baja precision, se perderia justo la informacion del error que
    # se esta buscando (cancelacion catastrofica).
    # ------------------------------------------------------------------
    r = b64 - A64 @ x
    res_norm = np.linalg.norm(r)                     # que tan lejos esta A@x de b
    err_norm = np.linalg.norm(x - x_true)             # que tan lejos esta x del resultado REAL
    print(f"{it:4d} | {str(r.dtype):>9} | {res_norm:15.3e} | {err_norm:15.3e}")

    if res_norm < tol:
        print("\nConvergio: el residuo ya es menor que la tolerancia.")
        break

    # ------------------------------------------------------------------
    # PASO 2: resolver el sistema de correccion A * dx = r
    # usando la MISMA factorizacion barata (float32) calculada una sola
    # vez al principio, sobre los datos originalmente almacenados en
    # float16. dx32 es una correccion APROXIMADA (baja precision).
    # ------------------------------------------------------------------
    dx32 = lu_solve((lu, piv), r.astype(np.float32))

    # ------------------------------------------------------------------
    # PASO 3: acumular la correccion EN FLOAT64.
    #   x_nuevo = x_actual + dx
    # Aqui es donde se "suma" la mejora a la solucion. Aunque dx se
    # calculo en precision baja, se promueve a float64 antes de sumarse,
    # para no perder la precision ya ganada en las iteraciones previas.
    # ------------------------------------------------------------------
    x = x + dx32.astype(np.float64)

# --- Comparacion final -------------------------------------------------
x_exacto_float64 = np.linalg.solve(A64, b64)  # "resultado real" resolviendo
                                               # todo en float64 de una sola vez
print(f"\nResultado obtenido (refinamiento mixto, datos guardados en float16): "
      f"||x - x_true|| = {np.linalg.norm(x - x_true):.3e}")
print(f"Resultado real (float64 puro, sin refinar): ||x_exacto64 - x_true|| = "
      f"{np.linalg.norm(x_exacto_float64 - x_true):.3e}")
print(f"Diferencia entre AMBOS resultados (mixto vs. float64 puro): "
      f"{np.linalg.norm(x - x_exacto_float64):.3e}")



Tamano del sistema: 400 x 400
dtype de A16 (almacenamiento original, formato MAS PEQUENO): float16
dtype de b16 (almacenamiento original): float16

(Solo para comparar resultados, se promueve una copia a float64:
 A64.dtype = float64, b64.dtype = float64)

>> Factorizando A (LAPACK no soporta float16, se promueve el minimo
   necesario para poder factorizar: float32)
   lu.dtype = float32

Solucion inicial x0 calculada en float32, promovida a float64
Error de x0 respecto al resultado REAL (x_true): 5.968e-03

iter |  dtype(r) |  ||r||=||b-Ax|| |  ||x - x_true||
-------------------------------------------------------
   1 |   float64 |       4.212e-03 |       5.968e-03
   2 |   float64 |       1.232e-09 |       5.967e-03
   3 |   float64 |       7.485e-12 |       5.967e-03

Convergio: el residuo ya es menor que la tolerancia.

Resultado obtenido (refinamiento mixto, datos guardados en float16): ||x - x_true|| = 5.967e-03
Resultado real (float64 puro, sin refinar): ||x_exacto64 - x_true|